### Import Libraries

In [1]:
import pandas as pd
import numpy as np

import joblib

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
model = joblib.load("../models/f1_xgboost_model.pkl")
FEATURES = joblib.load("../models/model_features.pkl")

### Load Historical Feature Dataset

We use this to:

- calculate rolling features
- compute momentum
- update standings

In [3]:
df = pd.read_csv("../data/processed/f1_eda_ready.csv")
df.head()

,race_id,year,round,race_date,race_name,circuit_id,driver_id,driver_code,driver_name,constructor_id,constructor_name,grid_position,finish_position,points,status_id,quali_position,winner,dnf_flag
0,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,20,vettel,Vettel,6,Ferrari,3,1,25.0,1,3.0,1,0
1,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,1,hamilton,Hamilton,131,Mercedes,1,2,18.0,1,1.0,0,0
2,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,8,raikkonen,Räikkönen,6,Ferrari,2,3,15.0,1,2.0,0,0
3,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,817,ricciardo,Ricciardo,9,Red Bull,8,4,12.0,1,5.0,0,0
4,989,2018,1,2018-03-25,Australian Grand Prix,albert_park,4,alonso,Alonso,1,McLaren,10,5,10.0,1,11.0,0,0


### Load Driver/Constructor Encodings

We must use SAME encodings as training.

In [4]:
# Load encoders 
le_driver = joblib.load( "../models/le_driver.pkl" ) 
le_constructor = joblib.load( "../models/le_constructor.pkl" ) 
le_circuit = joblib.load( "../models/le_circuit.pkl" ) 
print("Encoders loaded.")

Encoders loaded.


### Prediction Function

In [5]:
def build_race_features( quali_df, historical_df, circuit_id ): 
    """ Build prediction features from qualifying data. """ 
    prediction_rows = [] 
    for _, row in quali_df.iterrows(): 
        driver = row["driver_name"] 
        constructor = row["constructor_name"] 
        driver_hist = historical_df[ historical_df["driver_name"] == driver ].copy() 
        constructor_hist = historical_df[ historical_df["constructor_name"] == constructor ].copy() 
        # ------------------------------------------------- # Driver Features # ------------------------------------ 
        avg_finish_last3 = ( driver_hist.sort_values(by=["year", "round"]) ["finish_position"] .tail(3) .mean() ) 
        avg_points_last3 = ( driver_hist.sort_values(by=["year", "round"]) ["points"] .tail(3) .mean() ) 
        dnf_rate = ( ( driver_hist["finish_position"] > 15 ).mean() ) 
        # ------------------------------------------------- # Circuit Win Rate # -----------------------------------
        circuit_hist = driver_hist[ driver_hist["circuit_id"] == circuit_id ] 
        if len(circuit_hist) > 0: 
            win_rate = ( (circuit_hist["finish_position"] == 1) .mean() ) 
        else: win_rate = 0 
        # ------------------------------------------------- # Championship Momentum # ------------------------------ 
        driver_points_so_far = ( driver_hist["points"].sum() ) 
        constructor_points_so_far = ( constructor_hist["points"].sum() )
        constructor_avg_points_last3 = ( constructor_hist.groupby(["race_id"])["points"] .sum() .tail(3) .mean() ) 
        # ------------------------------------------------- # Encodings # ------------------------------------------------- 
        # Driver Encoding 
        try: 
            driver_encoded = le_driver.transform( [driver] )[0] 
        except: driver_encoded = -1
        # Constructor Encoding 
        try: 
            constructor_encoded = le_constructor.transform( [constructor] )[0] 
        except: constructor_encoded = -1
        # Circuit Encoding 
        try: 
            circuit_encoded = le_circuit.transform( [circuit_id] )[0] 
        except: circuit_encoded = -1
        # ------------------------------------------------- # Final Row # ------------------------------------------------- 
        prediction_rows.append({ "driver_name": driver, 
                                 "constructor_name": constructor, 
                                 "grid_position": row["grid_position"], 
                                 "quali_position": row["quali_position"], 
                                 "driver_avg_finish_last3": avg_finish_last3, 
                                 "driver_avg_points_last3": avg_points_last3, 
                                 "constructor_avg_points_last3": constructor_avg_points_last3, 
                                 "driver_win_rate_at_circuit": win_rate, 
                                 "driver_dnf_rate": dnf_rate, 
                                 "driver_points_so_far": driver_points_so_far, 
                                 "constructor_points_so_far": constructor_points_so_far, 
                                 "driver_encoded": driver_encoded, 
                                 "constructor_encoded": constructor_encoded, 
                                 "circuit_encoded": circuit_encoded 
                               })
    prediction_df = pd.DataFrame( prediction_rows ) 
    # --------------------------------------------- # Fill Missing Values # --------------------------------------------- 
    prediction_df[ "driver_avg_finish_last3" ] = ( prediction_df[ "driver_avg_finish_last3" ] .fillna(20) )
    remaining_features = [ "driver_avg_points_last3", "constructor_avg_points_last3", "driver_win_rate_at_circuit", "driver_dnf_rate", "driver_points_so_far", "constructor_points_so_far" ] 
    prediction_df[ remaining_features ] = ( prediction_df[ remaining_features ] .fillna(0) ) 
    return prediction_df

### Example

In [6]:
spain_quali = pd.DataFrame({
    "driver_name": [
        "Russell",
        "Hamilton",
        "Antonelli",
        "Norris",
        "Verstappen",
        "Hadjar",
        "Piastri",
        "Lawson",
        "Hulkenberg",
        "Leclerc",
        "Lindblad",
        "Bortoleto",
        "Colapinto",
        "Gasly",
        "Bearman",
        "Sainz",
        "Ocon",
        "Albon",
        "Perez",
        "Bottas",
        "Stroll",
        "Alonso"
    ],
    "constructor_name": [
        "Mercedes",
        "Ferrari",
        "Mercedes",
        "McLaren",
        "Red Bull",
        "Red Bull",
        "McLaren",
        "RB",
        "Audi",
        "Ferrari",
        "RB",
        "Audi",
        "Alpine",
        "Alpine",
        "Haas",
        "Williams",
        "Haas",
        "Williams",
        "Cadillac",
        "Cadillac",
        "Aston Martin",
        "Aston Martin"
    ],
    "grid_position": [
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
        21, 22
    ],
    "quali_position": [
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
        11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
        21, 22
    ]
})

In [7]:
spain_features = build_race_features( quali_df=spain_quali, historical_df=df, circuit_id="catalunya" )
spain_features.head()

,driver_name,constructor_name,grid_position,quali_position,driver_avg_finish_last3,driver_avg_points_last3,constructor_avg_points_last3,driver_win_rate_at_circuit,driver_dnf_rate,driver_points_so_far,constructor_points_so_far,driver_encoded,constructor_encoded,circuit_encoded
0,Russell,Mercedes,1,1,3.333333,15.666667,25.666667,0.000000,0.304688,664.0,3873.5,27,8,4
1,Hamilton,Ferrari,2,2,6.000000,10.000000,28.666667,0.571429,0.054054,2210.5,3005.5,12,4,4
2,Antonelli,Mercedes,3,3,20.000000,0.000000,25.666667,0.000000,0.000000,0.0,3873.5,-1,8,4
3,Norris,McLaren,4,4,5.666667,12.000000,19.333333,0.000000,0.109375,950.0,1708.0,22,7,4
4,Verstappen,Red Bull,5,5,4.000000,14.333333,14.666667,0.428571,0.107383,2491.5,3784.5,36,11,4


### Predict Win Probabilities

In [8]:
FEATURES

['grid_position',
 'quali_position',
 'driver_avg_finish_last3',
 'driver_avg_points_last3',
 'constructor_avg_points_last3',
 'driver_win_rate_at_circuit',
 'driver_dnf_rate',
 'driver_points_so_far',
 'constructor_points_so_far',
 'driver_encoded',
 'constructor_encoded',
 'circuit_encoded']

In [9]:
X_pred = spain_features[FEATURES]

spain_features["win_probability"] = (
    model.predict_proba(X_pred)[:,1]
)

In [10]:
spain_features

,driver_name,constructor_name,grid_position,quali_position,driver_avg_finish_last3,driver_avg_points_last3,constructor_avg_points_last3,driver_win_rate_at_circuit,driver_dnf_rate,driver_points_so_far,constructor_points_so_far,driver_encoded,constructor_encoded,circuit_encoded,win_probability
0,Russell,Mercedes,1,1,3.333333,15.666667,25.666667,0.000000,0.304688,664.0,3873.5,27,8,4,0.645864
1,Hamilton,Ferrari,2,2,6.000000,10.000000,28.666667,0.571429,0.054054,2210.5,3005.5,12,4,4,0.153710
2,Antonelli,Mercedes,3,3,20.000000,0.000000,25.666667,0.000000,0.000000,0.0,3873.5,-1,8,4,0.035774
3,Norris,McLaren,4,4,5.666667,12.000000,19.333333,0.000000,0.109375,950.0,1708.0,22,7,4,0.045447
4,Verstappen,Red Bull,5,5,4.000000,14.333333,14.666667,0.428571,0.107383,2491.5,3784.5,36,11,4,0.238945
5,Hadjar,Red Bull,6,6,20.000000,0.000000,14.666667,0.000000,0.000000,0.0,3784.5,-1,11,4,0.013405
6,Piastri,McLaren,7,7,6.666667,7.333333,19.333333,0.000000,0.108696,347.0,1708.0,24,7,4,0.022175
7,Lawson,RB,8,8,15.666667,0.000000,0.000000,0.000000,0.363636,6.0,0.0,18,-1,4,0.007125
8,Hulkenberg,Audi,9,9,20.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,-1,-1,4,0.007765
9,Leclerc,Ferrari,10,10,3.000000,15.000000,28.666667,0.000000,0.167785,1363.0,3005.5,19,4,4,0.065667


### Rank Drivers

In [11]:
predictions = (
    spain_features[
        [
            "driver_name",
            "constructor_name",
            "grid_position",
            "win_probability"
        ]
    ]
    .sort_values(
        by="win_probability",
        ascending=False
    )
)
predictions

,driver_name,constructor_name,grid_position,win_probability
0,Russell,Mercedes,1,0.645864
4,Verstappen,Red Bull,5,0.238945
1,Hamilton,Ferrari,2,0.153710
9,Leclerc,Ferrari,10,0.065667
3,Norris,McLaren,4,0.045447
15,Sainz,Williams,16,0.036964
2,Antonelli,Mercedes,3,0.035774
6,Piastri,McLaren,7,0.022175
21,Alonso,Aston Martin,22,0.016270
5,Hadjar,Red Bull,6,0.013405


In [12]:
winner = predictions.iloc[0]
print("🏆 Predicted Winner") 
print( f"{winner['driver_name']} " f"({winner['constructor_name']})" ) 
print( f"Win Probability: " f"{winner['win_probability']:.2%}" )

🏆 Predicted Winner
Russell (Mercedes)
Win Probability: 64.59%


### Testing on completed race

#### Step 1 — Create Monaco Qualifying Data

In [41]:
monaco_quali = pd.DataFrame({

    "driver_name": [
        "Kimi Antonelli",
        "Max Verstappen",
        "Lewis Hamilton",
        "Charles Leclerc",
        "Isack Hadjar",
        "George Russell",
        "Oscar Piastri",
        "Lando Norris",
        "Pierre Gasly",
        "Liam Lawson"
    ],

    "constructor_name": [
        "Mercedes",
        "Red Bull Racing",
        "Ferrari",
        "Ferrari",
        "Red Bull Racing",
        "Mercedes",
        "McLaren",
        "McLaren",
        "Alpine",
        "Racing Bulls"
    ],

    "grid_position": [
        1,2,3,4,5,6,7,8,9,10
    ],

    "quali_position": [
        1,2,3,4,5,6,7,8,9,10
    ]
})
monaco_quali

,driver_name,constructor_name,grid_position,quali_position
0,Kimi Antonelli,Mercedes,1,1
1,Max Verstappen,Red Bull Racing,2,2
2,Lewis Hamilton,Ferrari,3,3
3,Charles Leclerc,Ferrari,4,4
4,Isack Hadjar,Red Bull Racing,5,5
5,George Russell,Mercedes,6,6
6,Oscar Piastri,McLaren,7,7
7,Lando Norris,McLaren,8,8
8,Pierre Gasly,Alpine,9,9
9,Liam Lawson,Racing Bulls,10,10


#### Step 2 — Build Features

In [42]:
monaco_features = build_race_features( quali_df=monaco_quali, 
                                       historical_df=df, 
                                       circuit_id="monaco" )
monaco_features.head()

,driver_name,constructor_name,grid_position,quali_position,driver_avg_finish_last3,driver_avg_points_last3,constructor_avg_points_last3,driver_win_rate_at_circuit,driver_dnf_rate,driver_points_so_far,constructor_points_so_far,driver_encoded,constructor_encoded,circuit_encoded
0,Kimi Antonelli,Mercedes,1,1,20.0,0.0,25.666667,0,0.0,0.0,3873.5,-1,8,14
1,Max Verstappen,Red Bull Racing,2,2,20.0,0.0,0.000000,0,0.0,0.0,0.0,-1,-1,14
2,Lewis Hamilton,Ferrari,3,3,20.0,0.0,28.666667,0,0.0,0.0,3005.5,-1,4,14
3,Charles Leclerc,Ferrari,4,4,20.0,0.0,28.666667,0,0.0,0.0,3005.5,-1,4,14
4,Isack Hadjar,Red Bull Racing,5,5,20.0,0.0,0.000000,0,0.0,0.0,0.0,-1,-1,14


#### Step 3 — Predict Monaco

In [43]:
X_pred = monaco_features[FEATURES]
monaco_features['win_probability'] = (
    model.predict_proba(X_pred)[:,1]
)
monaco_features.head()

,driver_name,constructor_name,grid_position,quali_position,driver_avg_finish_last3,driver_avg_points_last3,constructor_avg_points_last3,driver_win_rate_at_circuit,driver_dnf_rate,driver_points_so_far,constructor_points_so_far,driver_encoded,constructor_encoded,circuit_encoded,win_probability
0,Kimi Antonelli,Mercedes,1,1,20.0,0.0,25.666667,0,0.0,0.0,3873.5,-1,8,14,0.078258
1,Max Verstappen,Red Bull Racing,2,2,20.0,0.0,0.000000,0,0.0,0.0,0.0,-1,-1,14,0.034323
2,Lewis Hamilton,Ferrari,3,3,20.0,0.0,28.666667,0,0.0,0.0,3005.5,-1,4,14,0.034659
3,Charles Leclerc,Ferrari,4,4,20.0,0.0,28.666667,0,0.0,0.0,3005.5,-1,4,14,0.018063
4,Isack Hadjar,Red Bull Racing,5,5,20.0,0.0,0.000000,0,0.0,0.0,0.0,-1,-1,14,0.011405


#### Step 4 — Rank Drivers

In [44]:
predictions = monaco_features[ [ "driver_name", 
                                 "constructor_name", 
                                 "grid_position", 
                                 "win_probability" ] ].sort_values(by='win_probability', ascending=False)
predictions

,driver_name,constructor_name,grid_position,win_probability
0,Kimi Antonelli,Mercedes,1,0.078258
2,Lewis Hamilton,Ferrari,3,0.034659
1,Max Verstappen,Red Bull Racing,2,0.034323
3,Charles Leclerc,Ferrari,4,0.018063
4,Isack Hadjar,Red Bull Racing,5,0.011405
5,George Russell,Mercedes,6,0.011285
7,Lando Norris,McLaren,8,0.007984
6,Oscar Piastri,McLaren,7,0.007967
9,Liam Lawson,Racing Bulls,10,0.007580
8,Pierre Gasly,Alpine,9,0.007506


In [45]:
actual_results = pd.DataFrame({
    "driver_name": [
        "Kimi Antonelli",
        "Lewis Hamilton",
        "Isack Hadjar"
    ],

    "actual_finish": [1,2,3]
})

comparison = predictions.merge(
    actual_results,
    on="driver_name",
    how="left"
)

comparison

,driver_name,constructor_name,grid_position,win_probability,actual_finish
0,Kimi Antonelli,Mercedes,1,0.078258,1.0
1,Lewis Hamilton,Ferrari,3,0.034659,2.0
2,Max Verstappen,Red Bull Racing,2,0.034323,NaN
3,Charles Leclerc,Ferrari,4,0.018063,NaN
4,Isack Hadjar,Red Bull Racing,5,0.011405,3.0
5,George Russell,Mercedes,6,0.011285,NaN
6,Lando Norris,McLaren,8,0.007984,NaN
7,Oscar Piastri,McLaren,7,0.007967,NaN
8,Liam Lawson,Racing Bulls,10,0.007580,NaN
9,Pierre Gasly,Alpine,9,0.007506,NaN
